# Extreme periods

Clustering finds *typical* behaviour, so a once-a-year peak day tends to be blended
into an average and lost. For capacity sizing or reliability that single day is
often the one that matters. `ExtremeConfig` forces chosen extremes to be kept
exactly, alongside the normal clustering.

In [ ]:
import pandas as pd
import plotly.io as pio

import tsam

pio.renderers.default = "notebook_connected"

raw = pd.read_csv("../data/testdata.csv", index_col=0, parse_dates=True)
data = raw.loc["2010-01-01":"2010-02-11"]  # six weeks of hourly data

## The peak gets averaged away

A plain aggregation clips the annual peak — the highest typical-day value sits well
below the real maximum:

In [ ]:
base = tsam.aggregate(data, n_clusters=6, period_duration="1D")
print(f"original peak Load:    {data['Load'].max():.1f}")
print(f"typical-period peak:   {base.cluster_representatives['Load'].max():.1f}")

## Keep it exactly

`ExtremeConfig` selects extreme **periods** to preserve. You can target the period
containing the single highest/lowest value (`max_value`/`min_value`) or the
highest/lowest period total (`max_period`/`min_period`):

In [ ]:
from tsam import ExtremeConfig

kept = tsam.aggregate(
    data,
    n_clusters=6,
    period_duration="1D",
    extremes=ExtremeConfig(method="new_cluster", max_value=["Load"]),
)
print(f"clusters: {base.n_clusters} -> {kept.n_clusters}")
print(f"typical-period peak now: {kept.cluster_representatives['Load'].max():.1f}")

## How the extreme enters: `method`

- **`new_cluster`** — add the extreme period as its own cluster (kept exactly,
  never blended). Recommended.
- **`append`** — add it as an extra typical period.
- **`replace`** — substitute the most similar existing cluster (cluster count
  unchanged).

In [ ]:
for m in ["new_cluster", "append", "replace"]:
    r = tsam.aggregate(
        data,
        n_clusters=6,
        period_duration="1D",
        extremes=ExtremeConfig(method=m, max_value=["Load"]),
    )
    print(
        f"{m:12s} -> {r.n_clusters} clusters, peak {r.cluster_representatives['Load'].max():.1f}"
    )

## Several extremes at once

Request as many as you need — e.g. the peak-demand day *and* the coldest day. A
period that is extreme for more than one criterion is added only once:

In [ ]:
multi = tsam.aggregate(
    data,
    n_clusters=6,
    period_duration="1D",
    extremes=ExtremeConfig(method="new_cluster", max_value=["Load"], min_value=["T"]),
)
print(f"clusters: {base.n_clusters} -> {multi.n_clusters}")

Extreme days stand in for just themselves, so they carry a small occurrence count
next to the typical clusters:

In [ ]:
kept.plot.cluster_counts()

## When do you need extremes at all?

Extremes are not free — `new_cluster` and `append` each add a typical period, and every added
period costs solve time. They earn that cost when **a single rare period sets a number you care
about**:

- **Capacity sizing and reliability.** If your model picks peaking capacity, grid limits or
  reserve from the worst hour, that hour must exist in the input. Clustering will otherwise
  average it away and undersize the system.
- **A binding constraint that only ever binds once.** A cold snap, a wind lull, a demand record.
- **Your peak is genuinely atypical.** That is exactly what makes clustering discard it.

**You probably do not need them when:**

- The answer is driven by **totals or averages** — annual energy, utilisation, average cost.
  Clustering plus [rescaling](../explanation/how-aggregation-works/05_rescaling.ipynb) already protects
  those.
- Your peak is not rare. If a dozen days are near the maximum, some cluster already represents
  them, and a representation that respects extremes is a lighter touch than a dedicated cluster.

### Three ways to keep a peak — which one?

Extremes are one of three levers, and they are not interchangeable:

| | What it does | Reach for it when |
|---|---|---|
| `ExtremeConfig` (here) | forces **one specific period** into the set, kept exactly | you can name the period that matters — the peak day, the coldest day |
| `maxoid` / `minmax_mean` [representation](representations.ipynb) | biases **every** representative toward its cluster's extreme | you want a generally conservative result, not one guaranteed day |
| `kmaxoids` [clustering](clustering_methods.ipynb) | spreads the **grouping** toward the edges of the data | the whole spread matters more than the average fit |

They compose, and the cheapest sufficient one usually wins. Note that `maxoid` fights with
[rescaling](../explanation/how-aggregation-works/05_rescaling.ipynb) — extreme representatives make
unrepresentative totals, so `preserve_column_means` may not fully converge. `ExtremeConfig` does
not have that problem: extreme clusters are excluded from rescaling by design.